# 04 — Multi-Contrast RGB Render Experiment (IXI T1+T2+PD)

**Idea**: IXI subjects have T1, T2, and PD (proton density) scans. Each contrast reflects different tissue properties:
- T1 → fat / myelin
- T2 → water content / inflammation  
- PD → proton density (tissue composition)

**Experiment**: Render a face image where R, G, B channels carry T1, T2, PD intensity respectively. Compare whether this multi-contrast render gives a higher Spearman ρ with chronological age than a standard T1-only render.

**Prerequisite**: T2 and PD must be registered to T1 space. IXI T2/PD are not pre-registered — we use a lightweight affine registration (ANTs or FSL FLIRT, or nibabel resampling if voxel grids match).

In [ ]:
import sys
from pathlib import Path

# ── CONFIG ──────────────────────────────────────────────────────────────────
IXI_T1_DIR   = Path("../data/ixi/T1")
IXI_T2_DIR   = Path("../data/ixi/T2")
IXI_PD_DIR   = Path("../data/ixi/PD")
IXI_META_XLS = Path("../data/ixi/IXI.xls")
FACEAGE_ROOT = Path("../vendor/FaceAge")
RENDERS_MC   = Path("../results/ixi_renders_mc")  # multi-contrast renders
RENDERS_T1   = Path("../results/ixi_renders")      # single T1 renders (from nb 03)
OUT_DIR      = Path("../results/ixi_multicontrast")
BYPASS_MTCNN = True
DEVICE       = "cpu"
# ────────────────────────────────────────────────────────────────────────────

RENDERS_MC.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(Path(".").resolve().parent))

In [ ]:
# ── 1. Find subjects with all 3 modalities ───────────────────────────────────
import re
import pandas as pd
import numpy as np

from src.utils import load_ixi_metadata

def ixi_id_from_path(p: Path) -> int | None:
    m = re.match(r"IXI(\d+)-", p.name)
    return int(m.group(1)) if m else None

t1_map = {ixi_id_from_path(f): f for f in IXI_T1_DIR.glob("IXI*-T1.nii.gz")}
t2_map = {ixi_id_from_path(f): f for f in IXI_T2_DIR.glob("IXI*-T2.nii.gz")}
pd_map = {ixi_id_from_path(f): f for f in IXI_PD_DIR.glob("IXI*-PD.nii.gz")}

common_ids = sorted(set(t1_map) & set(t2_map) & set(pd_map) - {None})
print(f"Subjects with T1+T2+PD: {len(common_ids)}")

meta = load_ixi_metadata(IXI_META_XLS)
df = meta[meta["IXI_ID"].isin(common_ids)].copy()
df["t1_path"] = df["IXI_ID"].map(lambda i: str(t1_map.get(i, "")))
df["t2_path"] = df["IXI_ID"].map(lambda i: str(t2_map.get(i, "")))
df["pd_path"] = df["IXI_ID"].map(lambda i: str(pd_map.get(i, "")))
df = df[df["t1_path"] != ""]
print(f"After metadata join: {len(df)} subjects")
print(df[["SITE", "AGE"]].describe())

In [ ]:
# ── 2. Registration check (are T2/PD in T1 space already?) ───────────────────
# IXI T2 and PD may have different voxel sizes. Check on one subject.
import nibabel as nib

sample = df.iloc[0]
t1 = nib.load(sample.t1_path)
t2 = nib.load(sample.t2_path)
pd_ = nib.load(sample.pd_path)

print(f"T1 : shape={t1.shape}  voxel={t1.header.get_zooms()}")
print(f"T2 : shape={t2.shape}  voxel={t2.header.get_zooms()}")
print(f"PD : shape={pd_.shape}  voxel={pd_.header.get_zooms()}")
print()
print("If shapes differ → registration needed before render_multicontrast.")
print("Option A: nibabel.processing.resample_from_to(t2, t1)  (affine-based, fast)")
print("Option B: FSL FLIRT / ANTs (more accurate, requires external tool)")

In [ ]:
# ── 3. Quick registration: resample T2 and PD to T1 space (nibabel) ──────────
from nibabel.processing import resample_from_to
from tqdm.notebook import tqdm
from src.render import render_multicontrast

df["render_mc_png"] = ""
df["render_mc_ok"] = False

REG_DIR = OUT_DIR / "registered"
REG_DIR.mkdir(exist_ok=True)

for idx, row in tqdm(df.iterrows(), total=len(df), desc="MC renders"):
    out_png = RENDERS_MC / f"IXI{row.IXI_ID:03d}_mc.png"
    if out_png.exists():
        df.at[idx, "render_mc_png"] = str(out_png)
        df.at[idx, "render_mc_ok"] = True
        continue

    # Resample T2 and PD into T1 space
    t1_img = nib.load(row.t1_path)
    t2_reg_path = REG_DIR / f"IXI{row.IXI_ID:03d}_T2_reg.nii.gz"
    pd_reg_path = REG_DIR / f"IXI{row.IXI_ID:03d}_PD_reg.nii.gz"

    if not t2_reg_path.exists():
        t2_reg = resample_from_to(nib.load(row.t2_path), t1_img)
        nib.save(t2_reg, str(t2_reg_path))
    if not pd_reg_path.exists():
        pd_reg = resample_from_to(nib.load(row.pd_path), t1_img)
        nib.save(pd_reg, str(pd_reg_path))

    ok = render_multicontrast(
        t1_path=row.t1_path,
        t2_path=str(t2_reg_path),
        pd_path=str(pd_reg_path),
        out_png=out_png,
    )
    df.at[idx, "render_mc_png"] = str(out_png)
    df.at[idx, "render_mc_ok"] = bool(ok)

print(f"MC renders OK: {df['render_mc_ok'].sum()} / {len(df)}")

In [ ]:
# ── 4. FaceAge on MC renders ─────────────────────────────────────────────────
from src.face_age import predict_age_batch

good = df[df["render_mc_ok"]].copy()
pngs_mc = [Path(p) for p in good["render_mc_png"]]

fa_mc = predict_age_batch(pngs_mc, FACEAGE_ROOT, bypass_mtcnn=BYPASS_MTCNN, device=DEVICE)
fa_mc_df = pd.DataFrame([{"render_mc_png": r["path"], "face_age_mc": r["age"]} for r in fa_mc])
good = good.merge(fa_mc_df, on="render_mc_png", how="left")
good["face_age_mc_gap"] = good["face_age_mc"] - good["AGE"]

In [ ]:
# ── 5. Load T1-only face ages from notebook 03 for comparison ────────────────
ixi_main_csv = Path("../results/ixi_main/ixi_all_predictions.csv")
if ixi_main_csv.exists():
    main_df = pd.read_csv(ixi_main_csv)[["IXI_ID", "face_age", "face_age_gap"]]
    good = good.merge(main_df.rename(columns={"face_age": "face_age_t1", "face_age_gap": "face_age_t1_gap"}),
                      on="IXI_ID", how="left")
else:
    print("Run notebook 03 first to get T1-only predictions for comparison.")

In [ ]:
# ── 6. Compare T1-only vs MC Spearman ρ with chronological age ──────────────
from scipy import stats
import matplotlib.pyplot as plt

analysis = good.dropna(subset=["face_age_mc"])

rho_mc, p_mc = stats.spearmanr(analysis["AGE"], analysis["face_age_mc"])
print(f"Multi-contrast RGB  ρ(age, face_age) = {rho_mc:.3f}  p={p_mc:.4g}")

if "face_age_t1" in analysis.columns:
    analysis_t1 = analysis.dropna(subset=["face_age_t1"])
    rho_t1, p_t1 = stats.spearmanr(analysis_t1["AGE"], analysis_t1["face_age_t1"])
    print(f"T1-only             ρ(age, face_age) = {rho_t1:.3f}  p={p_t1:.4g}")

# Visual comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, title, color in [
    (axes[0], "face_age_t1",  f"T1-only render  ρ={rho_t1:.3f}",          "steelblue"),
    (axes[1], "face_age_mc",  f"Multi-contrast  ρ={rho_mc:.3f}",           "mediumpurple"),
]:
    sub = analysis.dropna(subset=[col])
    ax.scatter(sub["AGE"], sub[col], alpha=0.4, s=15, color=color)
    m, b = np.polyfit(sub["AGE"], sub[col], 1)
    xs = np.linspace(sub["AGE"].min(), sub["AGE"].max(), 100)
    ax.plot(xs, m * xs + b, color="black", lw=1.5)
    ax.plot([20, 90], [20, 90], "--", color="gray", lw=1, label="y=x")
    ax.set_xlabel("Chronological age", fontsize=11)
    ax.set_ylabel("Predicted face age", fontsize=11)
    ax.set_title(title, fontsize=11)

plt.suptitle("IXI: T1-only vs Multi-contrast RGB face age", fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / "04_mc_vs_t1.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 7. Save ──────────────────────────────────────────────────────────────────
out_csv = OUT_DIR / "ixi_multicontrast_predictions.csv"
good.to_csv(out_csv, index=False)
print(f"Saved → {out_csv}")